In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load dataset
file_path = "all-vehicles-model-formatted-transformed_data.csv"
df = pd.read_csv(file_path)

# Drop unnecessary columns if exists
df = df.drop(columns=['Unnamed: 21'], errors='ignore')

# Handle missing values (fill with median for numerical columns)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Encode categorical variables
label_encoders = {}
categorical_cols = ['Make', 'Model', 'BaseModel']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Avoid division by zero in Rate calculation
df['Co2 Tailpipe For Fuel Type1'].replace(0, np.nan, inplace=True)
df['Annual Fuel Cost For Fuel Type1'].replace(0, np.nan, inplace=True)
df['Engine displacement'].replace(0, np.nan, inplace=True)

# Fill missing values with median
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Define Rate formula
alpha = 0.65
beta = 0.15
gamma = 0.1
delta = 0.05
epsilon = 0.2

df['Rate'] = (
    delta * (df['Combined Mpg For Fuel Type1'] / (df['Co2 Tailpipe For Fuel Type1'] + 1e-5)) +
    beta * (1 - df['Engine displacement'] / (df['Engine displacement'].max() + 1e-5)) +
    gamma * (1 - df['Annual Fuel Cost For Fuel Type1'] / (df['Annual Fuel Cost For Fuel Type1'].max() + 1e-5)) +
    alpha * (df['Make Value'] / (df['Make Value'].max() + 1e-5)) +
    epsilon * (df['Cylinders']) +
    0.04 * df['Year Value'] / (df['Year Value'].max() + 1e-5) +
    0.03 * df['Transmission Value'] / (df['Transmission Value'].max() + 1e-5) +
    0.03 * df['Drive Type Value'] / (df['Drive Type Value'].max() + 1e-5))

# Normalize Rate to 0 - 1000 scale
df['Rate'] = df['Rate'] * 1000 / df['Rate'].max()
df.dropna(subset=['Rate'], inplace=True)

# Define features and target
X = df[['Make', 'Model']]
y = df[['Rate', 'Cylinders', 'Transmission Value', 'Engine displacement', 'Vehicle Size Class Value']]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train model
model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Save the trained model
joblib.dump(model, 'vehicle_rate_model.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')
joblib.dump(scaler, 'scaler.pkl')
